In [40]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import classification_report
from skimage.feature import graycomatrix, graycoprops

# ── Konfigurasi ──────────────────────────────────────────────
BASE_DIR = Path("grape")

TRAIN_DIR = BASE_DIR / "train"
VALID_DIR = BASE_DIR / "valid"

TARGET_SIZE1 = 32
TARGET_SIZE2 = 64
TARGET_SIZE3 = 128

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Ambil nama kelas dari folder train
CLASS_NAMES = sorted([
    d.name for d in TRAIN_DIR.iterdir()
    if d.is_dir()
])

print("Kelas:", CLASS_NAMES)
print("Jumlah kelas:", len(CLASS_NAMES))

Kelas: ['Grape Esca (Black_Measles)', 'Grape Leaf blight (Isariopsis_Leaf_Spot)', 'grape leaf Healthy', 'grape leaf black rot']
Jumlah kelas: 4


In [41]:
def segment_leaf_hsv(image_bgr, min_area_ratio=0.01):
    """
    Segmentasi daun dengan HSV + pembersihan komponen.
    - Menggabungkan rentang warna daun (hijau/kuning/coklat/merah)
    - Menolak piksel low-saturation (background kusam)
    - Menyisakan komponen terbesar agar background tidak ikut
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    # Rentang warna daun (lebih ketat)
    lower_green = np.array([20, 35, 25])
    upper_green = np.array([95, 255, 255])

    lower_yellow = np.array([15, 40, 35])
    upper_yellow = np.array([35, 255, 255])

    lower_brown = np.array([5, 50, 20])
    upper_brown = np.array([20, 255, 220])

    lower_red1 = np.array([0, 50, 30])
    upper_red1 = np.array([12, 255, 255])
    lower_red2 = np.array([160, 50, 30])
    upper_red2 = np.array([180, 255, 255])

    # Mask warna
    mask_color  = cv2.inRange(hsv, lower_green, upper_green)
    mask_color |= cv2.inRange(hsv, lower_yellow, upper_yellow)
    mask_color |= cv2.inRange(hsv, lower_brown, upper_brown)
    mask_color |= cv2.inRange(hsv, lower_red1, upper_red1)
    mask_color |= cv2.inRange(hsv, lower_red2, upper_red2)

    # Tolak background abu/gelap (S dan V terlalu rendah)
    mask_sv = cv2.inRange(hsv, np.array([0, 35, 25]), np.array([180, 255, 255]))
    mask = cv2.bitwise_and(mask_color, mask_sv)

    # Morphological cleaning
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Keep komponen terbesar (leaf utama)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    refined = np.zeros_like(mask)

    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]  # abaikan background index 0
        largest_idx = 1 + np.argmax(areas)
        largest_area = stats[largest_idx, cv2.CC_STAT_AREA]

        h, w = mask.shape[:2]
        min_area = int(min_area_ratio * h * w)

        if largest_area >= min_area:
            refined[labels == largest_idx] = 255
        else:
            refined = mask.copy()
    else:
        refined = mask.copy()

    # Haluskan tepi akhir
    refined = cv2.medianBlur(refined, 5)
    result = cv2.bitwise_and(image_bgr, image_bgr, mask=refined)
    return result, refined

In [42]:
def apply_clahe(image_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    """
    Menerapkan CLAHE pada kanal L (Lightness) di ruang warna LAB.
    Kontras diperbaiki secara lokal tanpa mengubah pigmen warna asli.
    """
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_eq = clahe.apply(l_channel)

    lab_eq = cv2.merge([l_eq, a_channel, b_channel])
    result = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)
    return result

In [43]:
def zero_pad_and_resize(image_bgr, target_size=224):
    """
    1. Zero-padding → jadikan persegi (sisi = max(h, w))
    2. Resize       → target_size × target_size
    Aspect ratio asli dipertahankan; area tambahan diisi hitam.
    """
    h, w = image_bgr.shape[:2]
    max_side = max(h, w)

    # Buat canvas hitam persegi
    canvas = np.zeros((max_side, max_side, 3), dtype=np.uint8)

    # Letakkan gambar di tengah canvas
    y_offset = (max_side - h) // 2
    x_offset = (max_side - w) // 2
    canvas[y_offset:y_offset + h, x_offset:x_offset + w] = image_bgr

    # Resize ke target
    resized = cv2.resize(canvas, (target_size, target_size),
                         interpolation=cv2.INTER_AREA)
    return resized

In [44]:
def normalize_minmax(image_bgr):
    """
    Min-Max Normalization: pixel / 255.0
    Output bertipe float32 dengan rentang [0, 1].
    """
    return image_bgr.astype(np.float32) / 255.0

In [45]:
def augment_image(image):
    """
    Augmentasi geometris acak:
      - Rotasi 90°/180°/270°
      - Flip horizontal
      - Flip vertikal
      - Flip horizontal + vertikal
    Mengembalikan list gambar hasil augmentasi (TIDAK termasuk gambar asli).
    """
    augmented = []

    # Rotasi 90°, 180°, 270°
    for k in [1, 2, 3]:
        augmented.append(np.rot90(image, k=k).copy())

    # Flip horizontal
    augmented.append(np.flip(image, axis=1).copy())

    # Flip vertikal
    augmented.append(np.flip(image, axis=0).copy())

    # Flip horizontal + vertikal (setara rotasi 180° + flip)
    augmented.append(np.flip(image, axis=(0, 1)).copy())

    return augmented

# Ekstraksi Fitur Warna dengan Histogram HSV

In [46]:
def extract_hsv_hist_features(
    image_bgr,
    grid_size=(2, 2),   # 4-kuadran
    bins_h=8,
    bins_s=8,
    bins_v=8
):
    """
    Ekstraksi fitur warna HSV histogram per kuadran.
    Output: vektor fitur 1D.
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h, w = hsv.shape[:2]
    gh, gw = grid_size

    features = []

    for i in range(gh):
        for j in range(gw):
            y0 = int(i * h / gh)
            y1 = int((i + 1) * h / gh)
            x0 = int(j * w / gw)
            x1 = int((j + 1) * w / gw)

            cell = hsv[y0:y1, x0:x1]

            hist_h = cv2.calcHist([cell], [0], None, [bins_h], [0, 180])
            hist_s = cv2.calcHist([cell], [1], None, [bins_s], [0, 256])
            hist_v = cv2.calcHist([cell], [2], None, [bins_v], [0, 256])

            hist_h = cv2.normalize(hist_h, None).flatten()
            hist_s = cv2.normalize(hist_s, None).flatten()
            hist_v = cv2.normalize(hist_v, None).flatten()

            features.extend(hist_h)
            features.extend(hist_s)
            features.extend(hist_v)

    return np.array(features, dtype=np.float32)

# Ekstraksi Fitur Tekstur dengan GLCM

In [47]:
def extract_glcm_features(
    image_bgr,
    distances=(1, 2, 3),
    angles=(0, np.pi / 4, np.pi / 2, 3 * np.pi / 4),
    levels=32
):
    """
    Ekstraksi fitur tekstur menggunakan GLCM pada grayscale.
    Output: vektor fitur 1D.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Kuantisasi grayscale ke levels tertentu
    gray_q = np.floor(gray.astype(np.float32) * levels / 256.0).astype(np.uint8)
    gray_q = np.clip(gray_q, 0, levels - 1)

    glcm = graycomatrix(
        gray_q,
        distances=list(distances),
        angles=list(angles),
        levels=levels,
        symmetric=True,
        normed=True
    )

    props = ["contrast", "homogeneity", "energy", "correlation"]
    features = []

    for prop in props:
        values = graycoprops(glcm, prop)
        features.extend(values.flatten())

    return np.array(features, dtype=np.float32)

In [48]:
def preprocess_for_features(image_bgr):
    """
    Menggunakan preprocessing yang sudah ada:
    1. HSV masking
    2. CLAHE
    3. Zero-padding + resize
    4. Normalisasi
    Lalu dikembalikan ke uint8 agar aman untuk histogram dan GLCM.
    """
    seg, _ = segment_leaf_hsv(image_bgr)
    clahe = apply_clahe(seg)
    resized = zero_pad_and_resize(clahe, TARGET_SIZE3)
    normalized = normalize_minmax(resized)

    # Balik ke uint8 supaya cocok untuk ekstraksi fitur
    processed = (normalized * 255).astype(np.uint8)
    return processed

In [49]:
def extract_combined_features(image_bgr):
    hsv_features = extract_hsv_hist_features(image_bgr)
    glcm_features = extract_glcm_features(image_bgr)
    return np.concatenate([hsv_features, glcm_features]).astype(np.float32)

In [50]:
sample_class = CLASS_NAMES[0]
sample_dir = TRAIN_DIR / sample_class

sample_file = sorted([
    f for f in sample_dir.iterdir()
    if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
])[0]

image = cv2.imread(str(sample_file))

processed = preprocess_for_features(image)
features = extract_combined_features(processed)

print("File:", sample_file.name)
print("Shape original:", image.shape)
print("Shape processed:", processed.shape)
print("Jumlah fitur:", len(features))
print("5 fitur pertama:", features[:5])

File: Grape Esca (Black_Measles) (1).JPG
Shape original: (256, 256, 3)
Shape processed: (128, 128, 3)
Jumlah fitur: 144
5 fitur pertama: [0.9670365  0.23203777 0.10486322 0.         0.        ]


In [51]:
def generate_feature_names(
    grid_size=(2, 2),
    bins_h=8,
    bins_s=8,
    bins_v=8,
    distances=(1, 2, 3),
    angles=(0, np.pi/4, np.pi/2, 3*np.pi/4)
):

    feature_names = []

    # =====================================================
    # HSV HISTOGRAM
    # =====================================================

    gh, gw = grid_size

    quadrant = 1

    for i in range(gh):
        for j in range(gw):

            # H
            for b in range(bins_h):
                feature_names.append(
                    f"H_Q{quadrant}_B{b+1}"
                )

            # S
            for b in range(bins_s):
                feature_names.append(
                    f"S_Q{quadrant}_B{b+1}"
                )

            # V
            for b in range(bins_v):
                feature_names.append(
                    f"V_Q{quadrant}_B{b+1}"
                )

            quadrant += 1

    # =====================================================
    # GLCM
    # =====================================================

    props = [
        "contrast",
        "homogeneity",
        "energy",
        "correlation"
    ]

    angle_names = {
        0: "0",
        np.pi/4: "45",
        np.pi/2: "90",
        3*np.pi/4: "135"
    }

    for prop in props:

        for d in distances:

            for a in angles:

                feature_names.append(
                    f"{prop}_d{d}_a{angle_names[a]}"
                )

    return feature_names

In [52]:
feature_names = generate_feature_names()

print("Jumlah nama fitur:", len(feature_names))

print("\n10 nama fitur pertama:")
print(feature_names[:10])

Jumlah nama fitur: 144

10 nama fitur pertama:
['H_Q1_B1', 'H_Q1_B2', 'H_Q1_B3', 'H_Q1_B4', 'H_Q1_B5', 'H_Q1_B6', 'H_Q1_B7', 'H_Q1_B8', 'S_Q1_B1', 'S_Q1_B2']


In [53]:
train_rows = []

for class_name in CLASS_NAMES:

    class_dir = TRAIN_DIR / class_name

    files = sorted([
        f for f in class_dir.iterdir()
        if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    ])

    print(f"Processing train: {class_name}")

    for file in tqdm(files):

        image = cv2.imread(str(file))
        if image is None:
            continue

        processed = preprocess_for_features(image)
        features = extract_combined_features(processed)

        row = {
            "filename": file.name,
            "label": class_name
        }

        for name, val in zip(feature_names, features):
            row[name] = float(val)

        train_rows.append(row)

print("Total train:", len(train_rows))

train_df = pd.DataFrame(train_rows)

Processing train: Grape Esca (Black_Measles)


100%|██████████| 1920/1920 [00:10<00:00, 187.60it/s]


Processing train: Grape Leaf blight (Isariopsis_Leaf_Spot)


100%|██████████| 1722/1722 [00:09<00:00, 187.27it/s]


Processing train: grape leaf Healthy


100%|██████████| 1749/1749 [00:15<00:00, 115.54it/s]


Processing train: grape leaf black rot


100%|██████████| 1944/1944 [00:12<00:00, 150.15it/s]


Total train: 7335


In [54]:
train_df

,filename,label,H_Q1_B1,H_Q1_B2,H_Q1_B3,H_Q1_B4,H_Q1_B5,H_Q1_B6,H_Q1_B7,H_Q1_B8,...,correlation_d1_a90,correlation_d1_a135,correlation_d2_a0,correlation_d2_a45,correlation_d2_a90,correlation_d2_a135,correlation_d3_a0,correlation_d3_a45,correlation_d3_a90,correlation_d3_a135
0,Grape Esca (Black_Measles) (1).JPG,Grape Esca (Black_Measles),0.967036,0.232038,0.104863,0.000000,0.000000,0.0,0.0,0.001594,...,0.900290,0.855121,0.787892,0.847899,0.824649,0.855121,0.725054,0.747012,0.775561,0.758186
1,Grape Esca (Black_Measles) (10).JPG,Grape Esca (Black_Measles),0.938879,0.299613,0.169481,0.003107,0.000690,0.0,0.0,0.002071,...,0.893771,0.851236,0.790222,0.837661,0.789238,0.851236,0.712149,0.708121,0.716997,0.720409
2,Grape Esca (Black_Measles) (100).JPG,Grape Esca (Black_Measles),0.991178,0.070245,0.112393,0.000287,0.000000,0.0,0.0,0.000287,...,0.908502,0.883942,0.815515,0.853537,0.823443,0.883942,0.757666,0.758409,0.759253,0.792105
3,Grape Esca (Black_Measles) (1000).JPG,Grape Esca (Black_Measles),0.973810,0.180136,0.138664,0.000000,0.000000,0.0,0.0,0.004116,...,0.896680,0.859054,0.842530,0.883147,0.827962,0.859054,0.794676,0.810506,0.779990,0.778465
4,Grape Esca (Black_Measles) (1001).JPG,Grape Esca (Black_Measles),0.975160,0.076580,0.207816,0.002162,0.000618,0.0,0.0,0.002470,...,0.896722,0.883637,0.842390,0.858836,0.827428,0.883637,0.794442,0.777542,0.779349,0.810359
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7330,grape leaf black rot (995).JPG,grape leaf black rot,0.958318,0.278899,0.061765,0.000000,0.000000,0.0,0.0,0.005094,...,0.897671,0.853867,0.815141,0.870165,0.814214,0.853867,0.752501,0.782047,0.755372,0.751544
7331,grape leaf black rot (996).JPG,grape leaf black rot,0.953609,0.219460,0.205125,0.000000,0.000000,0.0,0.0,0.019796,...,0.898193,0.871046,0.815452,0.854018,0.814902,0.871046,0.752651,0.752094,0.756269,0.783151
7332,grape leaf black rot (997).JPG,grape leaf black rot,0.936861,0.054136,0.334470,0.086548,0.000000,0.0,0.0,0.000345,...,0.921733,0.868390,0.786349,0.860941,0.848173,0.868390,0.713061,0.754304,0.792391,0.766816
7333,grape leaf black rot (998).JPG,grape leaf black rot,0.834679,0.043646,0.546473,0.052663,0.000000,0.0,0.0,0.000000,...,0.911427,0.887171,0.863894,0.887211,0.849302,0.887171,0.822963,0.825693,0.809569,0.820453


In [55]:
valid_rows = []

for class_name in CLASS_NAMES:

    class_dir = VALID_DIR / class_name

    files = sorted([
        f for f in class_dir.iterdir()
        if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    ])

    print(f"Processing valid: {class_name}")

    for file in tqdm(files):

        image = cv2.imread(str(file))
        if image is None:
            continue

        processed = preprocess_for_features(image)
        features = extract_combined_features(processed)

        row = {
            "filename": file.name,
            "label": class_name
        }

        for name, val in zip(feature_names, features):
            row[name] = float(val)

        valid_rows.append(row)

print("Total valid:", len(valid_rows))

valid_df = pd.DataFrame(valid_rows)

Processing valid: Grape Esca (Black_Measles)


100%|██████████| 480/480 [00:02<00:00, 178.61it/s]


Processing valid: Grape Leaf blight (Isariopsis_Leaf_Spot)


100%|██████████| 430/430 [00:02<00:00, 188.30it/s]


Processing valid: grape leaf Healthy


100%|██████████| 435/435 [00:02<00:00, 150.88it/s]


Processing valid: grape leaf black rot


100%|██████████| 480/480 [00:02<00:00, 165.46it/s]

Total valid: 1825


In [56]:
valid_df

,filename,label,H_Q1_B1,H_Q1_B2,H_Q1_B3,H_Q1_B4,H_Q1_B5,H_Q1_B6,H_Q1_B7,H_Q1_B8,...,correlation_d1_a90,correlation_d1_a135,correlation_d2_a0,correlation_d2_a45,correlation_d2_a90,correlation_d2_a135,correlation_d3_a0,correlation_d3_a45,correlation_d3_a90,correlation_d3_a135
0,Grape Esca (Black_Measles) (1).JPG,Grape Esca (Black_Measles),0.968124,0.249277,0.022273,0.000000,0.000000,0.000000,0.000000,0.010069,...,0.904500,0.836408,0.743694,0.835856,0.802120,0.836408,0.667449,0.708936,0.734126,0.724151
1,Grape Esca (Black_Measles) (10).JPG,Grape Esca (Black_Measles),0.905277,0.064440,0.419206,0.023559,0.005543,0.001039,0.000000,0.000000,...,0.901989,0.845553,0.753670,0.816765,0.825407,0.845553,0.717967,0.736646,0.772402,0.759155
2,Grape Esca (Black_Measles) (100).JPG,Grape Esca (Black_Measles),0.957915,0.132432,0.254677,0.000000,0.000000,0.000000,0.000000,0.000986,...,0.909273,0.825924,0.764309,0.844849,0.842468,0.825924,0.730671,0.768072,0.796940,0.748956
3,Grape Esca (Black_Measles) (101).JPG,Grape Esca (Black_Measles),0.925738,0.310249,0.215779,0.000000,0.000000,0.000000,0.000000,0.013956,...,0.873146,0.833454,0.825206,0.860676,0.783206,0.833454,0.767775,0.768153,0.729926,0.744719
4,Grape Esca (Black_Measles) (102).JPG,Grape Esca (Black_Measles),0.954286,0.203985,0.218459,0.000000,0.000000,0.000000,0.000000,0.002020,...,0.892914,0.830719,0.801579,0.868556,0.803444,0.830719,0.747219,0.776372,0.748942,0.739076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1820,grape leaf black rot (95).JPG,grape leaf black rot,0.935546,0.034889,0.351476,0.001292,0.000000,0.000000,0.000000,0.000000,...,0.925212,0.891964,0.857134,0.897597,0.869919,0.891964,0.810454,0.828626,0.825043,0.815469
1821,grape leaf black rot (96).JPG,grape leaf black rot,0.818840,0.574008,0.002049,0.000000,0.000000,0.000000,0.000341,0.003415,...,0.911171,0.868683,0.797592,0.853514,0.834156,0.868683,0.730252,0.754226,0.776144,0.772150
1822,grape leaf black rot (97).JPG,grape leaf black rot,0.928153,0.108746,0.354984,0.026321,0.000000,0.000000,0.000000,0.000346,...,0.902379,0.849151,0.823784,0.880562,0.825441,0.849151,0.780191,0.801950,0.779016,0.771161
1823,grape leaf black rot (98).JPG,grape leaf black rot,0.837397,0.080245,0.540670,0.001783,0.000000,0.000000,0.000000,0.000713,...,0.931738,0.901179,0.821805,0.870914,0.862485,0.901179,0.762207,0.775698,0.808578,0.823300


In [57]:
X_train = train_df.drop(columns=["filename", "label"])
y_train = train_df["label"]

X_test = valid_df.drop(columns=["filename", "label"])
y_test = valid_df["label"]

In [58]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [59]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_scaled, y_train)

KNeighborsClassifier()

In [60]:
from sklearn.metrics import accuracy_score

# Prediksi data train
y_train_pred = knn.predict(X_train_scaled)

# Prediksi data test
y_test_pred = knn.predict(X_test_scaled)

# Accuracy
train_acc = accuracy_score(y_train, y_train_pred)

test_acc = accuracy_score(y_test, y_test_pred)

print("Training Accuracy :", train_acc)
print("Testing Accuracy  :", test_acc)

Training Accuracy : 0.9466939331970007
Testing Accuracy  : 0.9161643835616439
